In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets
import torchvision.transforms.v2 as transforms
import torchvision.models as models

In [3]:
# 0. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

print(f'model:{dir(model)}')
print(f'model:{model.fc}')

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)

model:['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_impl', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_make_layer', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_norm_laye

In [ ]:
########################################################
# 4. 손실 함수, 옵티마이저 설정
########################################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

########################################################
# 5. 학습 및 검증 루프 (Training & Validation Loop)
########################################################
epochs = 5

for epoch in range(epochs):
    # --- [Training Phase (증강 적용)] ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # --- [Validation Phase (증강 미적용)] ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc*100:.2f}%")

########################################################
# 6. 최종 테스트 평가 (Test Phase)
########################################################
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_epoch_loss = test_loss / test_total
test_epoch_acc = test_correct / test_total
print(f"\n[Test Result] Loss: {test_epoch_loss:.4f}, Accuracy: {test_epoch_acc*100:.2f}%")

In [ ]:
from pathlib import Path
import matplotlib as plt
import numpy as np

save_dir = Path("./augmented_mnist")
save_dir.mkdir(parents=True, exist_ok=True)

row, col = 2, 4

transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=0.3,
        scale=(0.8, 1.2)
    )
])

fig, axes = plt.subplots(row, col, figsize=(8, 4))
axes = np.asarray(axes).reshape(row, col)

for idx in range(row * col):
    image, label = mnist_train_raw[idx]

    augmented = transform(image)

    # PIL 이미지로 저장
    save_path = save_dir / f"aug_{idx+1:02d}_label_{label}.png"
    augmented.save(save_path)

    axes[idx // col, idx % col].imshow(augmented, cmap="Greys")
    axes[idx // col, idx % col].axis("off")
    axes[idx // col, idx % col].set_title(f"label={label}")

plt.tight_layout()
plt.show()

print(f"저장 위치 : {save_dir.resolve()}")
print("저장된 파일 수 :", len(list(save_dir.glob("*.png"))))